# 📷 09 · Electronic Imaging: Detectors and the Digital Pipeline
### From a photon landing on silicon to a number in an array

**Part B · Seeing like a microscope** · Module 09 of 37 · *BiologyToBioimage*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/09_electronic_imaging_detectors_and_the_digital_pipeline.ipynb)

| | |
|---|---|
| **Hardware target** | CPU only (no GPU needed) |
| **Core frameworks** | NumPy, Matplotlib |
| **Prerequisites** | Modules 03–08 |
| **Estimated time** | 75–90 minutes |
| **Instrumentation topics covered** | Photon shot noise and why SNR scales as the square root of photon count; the camera detection chain (quantum efficiency, dark current, read noise, gain); comparing low-read-noise and high-read-noise sensors; bit-depth quantization; the fixed photon budget trade-off in live-cell time-lapse imaging |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every photon count, noise term and quantized image here is SIMULATED; no real camera's dark frame or calibration file is read from disk.

## 🎯 Learning objectives

1. Explain photon shot noise as a fundamental consequence of the discreteness of light (Poisson
   statistics), not a flaw in any particular camera, and why signal-to-noise ratio scales as the
   square root of photon count.
2. Simulate a full camera detection chain from incident photons to digital counts, and identify the
   separate contributions of quantum efficiency, dark current, and read noise.
3. Compare simulated low-read-noise (EMCCD/sCMOS-like) and high-read-noise (older CCD-like) sensors
   at low and high light levels, and explain why read noise matters most when photon counts are
   small.
4. Simulate bit-depth quantization and explain when it becomes a visible, limiting source of error
   relative to photon shot noise, and when it does not.
5. Simulate the fixed photon-budget trade-off in live-cell time-lapse imaging, and explain why more
   time points inevitably means lower per-frame signal-to-noise for a sample that only tolerates a
   fixed total amount of light exposure.

## 🗺️ Contents

1. Photon shot noise: light itself is grainy
2. The camera detection chain: quantum efficiency, dark current, read noise, gain
3. Sensor comparison: when does read noise matter?
4. Bit-depth quantization: how many grey levels are actually needed?
5. The photon budget: live-cell time-lapse's fundamental trade-off

## 🧫 The biological question

Every optical trick from Modules 06-08 for getting a cleaner, more selective image comes at a cost
measured in photons: a narrower confocal pinhole, a dim evanescent TIRF field, a sparsely-blinking
localization-microscopy fluorophore all mean fewer photons reach the detector per unit time. Whether
that remaining trickle of photons produces a usable image or an unusably noisy mess is decided
entirely by the electronics covered in this module — identical optics can look pristine on an
expensive low-noise camera and unusable on a budget one, and identical electronics can look pristine
at one light level and hopeless at another. Understanding this chain is what lets a microscopist
predict, before ever touching the instrument, whether a proposed experiment is even physically
possible with the light available.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
print('No GPU is needed anywhere in this notebook — everything is plain NumPy/Matplotlib.')

In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
SEED = 9
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11

### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.1; copied so this notebook runs alone) ──
import numpy as np

def sensor_noise_model(incident_photons: np.ndarray, qe: float = 0.7, read_noise_e: float = 2.0,
                        dark_current_e: float = 0.05, gain: float = 1.0,
                        rng: np.random.Generator | None = None) -> np.ndarray:
    """A simplified camera detection chain: Poisson-sample the photoelectrons actually detected
    (``incident_photons * qe``, the quantum efficiency), add Poisson dark-current electrons, add
    Gaussian read noise (``read_noise_e``, electrons RMS, independent of signal), and apply an
    output ``gain`` (digital counts per electron). Use a low ``read_noise_e`` (~1-2, an EMCCD or
    sCMOS) vs. a higher one (~8-15, an older CCD) to compare sensor types."""
    rng = rng or np.random.default_rng(0)
    photoelectrons = rng.poisson(np.clip(incident_photons, 0, None) * qe).astype(float)
    dark = rng.poisson(dark_current_e, size=incident_photons.shape).astype(float)
    signal_e = photoelectrons + dark
    read_noise = rng.normal(0.0, read_noise_e, size=incident_photons.shape)
    return (signal_e + read_noise) * gain


def quantize_bits(image: np.ndarray, n_bits: int) -> np.ndarray:
    """Quantise an image in [0, 1] to ``n_bits`` of digitisation (2**n_bits grey levels)."""
    levels = 2 ** n_bits
    return np.floor(np.clip(image, 0, 1) * (levels - 1)) / (levels - 1)


def photon_budget_tradeoff(total_photon_budget: float, n_frames: np.ndarray) -> dict:
    """The fundamental live-cell imaging trade-off: a fixed total photon budget (set by how much
    bleaching/phototoxicity the sample tolerates) must be divided across ``n_frames`` time
    points. More frames (a longer or finer-grained time-lapse) means fewer photons per frame —
    and therefore lower signal-to-noise, since shot-noise SNR scales as sqrt(photons per frame).
    Returns ``photons_per_frame`` and the resulting ``snr_per_frame`` for each entry of
    ``n_frames``."""
    photons_per_frame = total_photon_budget / n_frames
    snr_per_frame = np.sqrt(photons_per_frame)
    return dict(photons_per_frame=photons_per_frame, snr_per_frame=snr_per_frame)

---
## 1 · Photon shot noise: light itself is grainy

Light arrives as discrete photons, and the number actually detected in any fixed interval is
fundamentally random, following Poisson statistics — even from a perfectly steady source and a
perfectly noiseless detector. For a Poisson process with mean $N$, the standard deviation is
$\sqrt{N}$, so the *relative* noise is $\sqrt{N}/N = 1/\sqrt{N}$: dimmer signals are not just noisier
in an absolute sense, they are proportionally *much* noisier. This single fact, called **shot noise**,
sets a hard floor on image quality that exists before any camera electronics are even considered.

In [ ]:
# ── 1.1 confirm the 1/sqrt(N) relative-noise law directly by simulation ──
mean_photon_counts = np.array([5, 20, 100, 500, 2000, 10000])
n_trials = 20000
relative_noise_empirical = []
for mean_n in mean_photon_counts:
    samples = rng.poisson(mean_n, size=n_trials)
    relative_noise_empirical.append(samples.std() / samples.mean())
relative_noise_empirical = np.array(relative_noise_empirical)
relative_noise_theory = 1 / np.sqrt(mean_photon_counts)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.loglog(mean_photon_counts, relative_noise_empirical, 'o', label='empirical (simulated)', ms=7)
ax.loglog(mean_photon_counts, relative_noise_theory, '--', label='theory: 1/sqrt(N)', color='gray')
ax.set_xlabel('mean photon count N'); ax.set_ylabel('relative noise (std / mean)')
ax.set_title('Shot noise: dimmer signals are proportionally noisier')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── 1.2 the same effect, shown visually on a flat test object at different photon budgets ──
size = 120
yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
disc = (np.hypot(yy - size / 2, xx - size / 2) < size * 0.3).astype(float)

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, mean_photons in zip(axes, [5, 50, 500, 5000]):
    noisy = rng.poisson(disc * mean_photons)
    ax.imshow(noisy, cmap='inferno')
    ax.set_title(f'mean = {mean_photons} photons/px', fontsize=9)
    ax.axis('off')
plt.suptitle('The identical disc, imaged with increasing photon budgets')
plt.tight_layout(); plt.show()

🔍 **What just happened?** The disc is barely distinguishable from background noise at 5
photons per pixel on average, and becomes progressively cleaner purely by increasing photon count —
no denoising algorithm was applied anywhere in this figure. This is the baseline every later module
on denoising (Module 24 and others) has to work against: shot noise is not a correctable flaw, it is
a statistical property of light itself, and denoising can only ever exploit redundancy in the image
to estimate the underlying signal more precisely, never eliminate the randomness retroactively.

✏️ **Your turn.** Add a fifth panel at `mean_photons = 50000`. Does the disc's edge still look
perfectly sharp, or is there a visible residual graininess even at this much higher photon count?

### 📘 Lecture notes — shot noise is a property of light, not of any particular camera

**Definition.** Shot noise is the Poisson-distributed randomness in the number of photons detected in
a fixed interval from a source of constant average intensity; its relative size scales as
$1/\sqrt{N}$, independent of any instrument's quality.

**Worked example.** Doubling exposure time (and hence photon count) improves shot-noise-limited SNR
by only a factor of $\sqrt{2} \approx 1.41$, not a factor of 2 — a quick mental-math check worth
having before assuming a longer exposure will "fix" a noisy image proportionally.

**One memorable point.** *A "noisy image" is not always evidence of a bad camera — at low light
levels, shot noise alone can dominate even a theoretically perfect, noiseless detector.*

---
## 2 · The camera detection chain: quantum efficiency, dark current, read noise, gain

Shot noise describes the light itself; a real camera adds several further, independent stages
before a final digital number is produced. **Quantum efficiency (QE)** is the fraction of incident
photons that actually get converted to a detectable photoelectron (not every photon is absorbed and
converted). **Dark current** is a small, also-Poisson-distributed trickle of thermally-generated
electrons, present even with no light at all. **Read noise** is Gaussian electronic noise added by
the circuitry that measures each pixel's accumulated charge, independent of the signal level. Finally
a **gain** factor converts electrons to the digital counts actually stored in the image file.
`sensor_noise_model` chains all four stages together in order.

In [ ]:
# ── 2.1 build up the detection chain stage by stage ──
incident = disc * 300.0  # photons/pixel incident on the sensor

stage_photoelectrons = sensor_noise_model(incident, qe=0.7, read_noise_e=0.0, dark_current_e=0.0, rng=rng)
stage_plus_dark = sensor_noise_model(incident, qe=0.7, read_noise_e=0.0, dark_current_e=0.05, rng=rng)
stage_full = sensor_noise_model(incident, qe=0.7, read_noise_e=2.0, dark_current_e=0.05, rng=rng)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
titles = ['Photoelectrons only\n(shot noise, QE=0.7)', '+ dark current', '+ read noise (full chain)']
for ax, img, title in zip(axes, [stage_photoelectrons, stage_plus_dark, stage_full], titles):
    ax.imshow(img, cmap='inferno', vmin=0, vmax=incident.max() * 0.7)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each stage adds a visually distinct kind of degradation: the first
panel already shows shot noise scaled down by QE; dark current adds a faint, spatially random
background even where `incident` is zero; read noise adds a further layer of background graininess on
top, visible everywhere including the dark background outside the disc. All three sources are present
simultaneously in any real image — this figure exists to make each one individually identifiable.

✏️ **Your turn.** Lower `qe` to `0.3` in `stage_photoelectrons` (and keep the other two stages
matched) and compare the disc's visibility. Why does lowering QE have an effect that looks similar to
simply lowering the incident photon count, even though they are physically different things?

---
## 3 · Sensor comparison: when does read noise matter?

Read noise is a *fixed* amount per pixel, independent of signal level — which means its relative
importance compared to shot noise changes dramatically with how much light is available. At high
photon counts, shot noise (which grows with the signal) dominates read noise (which does not); at low
photon counts, a camera with high read noise can be overwhelmed by its own electronics, while a
low-read-noise camera (such as a modern sCMOS or an EMCCD) still reveals real structure.

In [ ]:
# ── 3.1 compare a low-read-noise and a high-read-noise sensor, at low and high light levels ──
sensors = {'low read noise (sCMOS-like, 1.5 e-)': 1.5, 'high read noise (older CCD-like, 12 e-)': 12.0}
light_levels = {'dim signal (20 photons/px)': 20.0, 'bright signal (2000 photons/px)': 2000.0}

fig, axes = plt.subplots(2, 2, figsize=(9, 7.5))
for i, (sensor_name, read_noise_e) in enumerate(sensors.items()):
    for j, (level_name, mean_photons) in enumerate(light_levels.items()):
        incident_level = disc * mean_photons
        out = sensor_noise_model(incident_level, qe=0.7, read_noise_e=read_noise_e,
                                  dark_current_e=0.05, rng=rng)
        axes[i, j].imshow(out, cmap='inferno', vmin=0, vmax=mean_photons * 0.7)
        axes[i, j].set_title(f'{sensor_name}\n{level_name}', fontsize=8)
        axes[i, j].axis('off')
plt.tight_layout(); plt.show()

🔍 **What just happened?** At the bright signal level (right column), both sensors produce a
clearly visible disc — shot noise dominates for both, so the extra read noise barely matters. At the
dim signal level (left column), the high-read-noise sensor's image is noticeably more degraded than
the low-read-noise sensor's — exactly the gap that justifies the substantial extra cost of a
low-read-noise camera for photon-starved applications like single-molecule localization microscopy
(Module 08) or deep two-photon imaging, but which would barely matter for a bright, high-signal
bulk-fluorescence experiment.

✏️ **Your turn.** Add a third light level, `'very dim (3 photons/px)'`, and re-run. Does the gap
between the two sensors widen or narrow further at this even lower light level?

---
## 4 · Bit-depth quantization: how many grey levels are actually needed?

After all of the analog noise sources above are combined, the result is still a continuous
(analog) value that must be digitized into a finite number of grey levels — typically 8, 12, or 16
bits in practice. Quantizing too coarsely introduces visible **banding**: artificial, stair-step
jumps in intensity that are not present in the true signal. But quantizing much more finely than the
noise already present is also pointless — once the quantization step is much smaller than the shot
noise standard deviation at a given signal level, finer bit depth stops improving anything
perceptible.

In [ ]:
# ── 4.1 quantization banding at a few bit depths, on a smooth gradient ──
gradient = np.tile(np.linspace(0, 1, 256), (60, 1))

fig, axes = plt.subplots(4, 1, figsize=(8, 5))
for ax, n_bits in zip(axes, [8, 4, 3, 2]):
    ax.imshow(quantize_bits(gradient, n_bits), cmap='gray', vmin=0, vmax=1, aspect='auto')
    ax.set_title(f'{n_bits}-bit ({2 ** n_bits} grey levels)', fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.2 when does the quantization step become smaller than the shot-noise floor? ──
mean_photons_range = np.array([10, 50, 200, 1000, 5000, 20000])
shot_noise_relative = 1 / np.sqrt(mean_photons_range)

for n_bits in [8, 12, 16]:
    quantization_step_relative = 1 / (2 ** n_bits - 1)
    print(f"{n_bits}-bit quantization step (relative): {quantization_step_relative:.2e}")
print()
for mean_photons, rel_noise in zip(mean_photons_range, shot_noise_relative):
    print(f'At {mean_photons:>6d} photons/px, shot-noise relative std = {rel_noise:.2e} '
          f'({"8-bit step is already coarser" if rel_noise < 1/255 else "8-bit step still smaller than shot noise"})')

🔍 **What just happened?** The smooth gradient visibly bands at 4, 3, and 2 bits — artificial
steps appear that are not present in the true continuous signal. The printed comparison shows that at
low photon counts, shot noise's relative size is actually *larger* than an 8-bit quantization step,
meaning additional bit depth there would capture essentially nothing but noise; only at the highest
photon counts does shot noise shrink below the 8-bit step, which is exactly why scientific cameras
commonly default to 12- or 16-bit digitization for brighter, shot-noise-limited signals.

✏️ **Your turn.** At roughly what `mean_photons` value does the printed comparison flip from
"8-bit step is already coarser" to "8-bit step still smaller than shot noise"? Does this match where
you would expect banding artifacts to actually start mattering in a real 8-bit image?

---
## 5 · The photon budget: live-cell time-lapse's fundamental trade-off

A live specimen can only tolerate a limited total amount of excitation light before
photobleaching or phototoxicity (Module 10) becomes unacceptable — a fixed **photon budget** for the
entire experiment. Splitting that fixed budget across more time points (a longer movie, or finer
temporal sampling) necessarily means fewer photons are available per individual frame, and because
shot-noise-limited SNR scales as $\sqrt{\text{photons per frame}}$, each additional time point comes
at a direct, unavoidable cost to per-frame image quality.

In [ ]:
# ── 5.1 the photon-budget trade-off: more time points, less SNR per frame ──
total_budget = 2_000_000.0  # total photons the sample can tolerate across the whole movie
n_frames_range = np.array([5, 10, 20, 50, 100, 200, 500])
result = photon_budget_tradeoff(total_budget, n_frames_range)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].loglog(n_frames_range, result['photons_per_frame'], 'o-', color='teal')
axes[0].set_xlabel('number of time points in the movie'); axes[0].set_ylabel('photons per frame')
axes[0].set_title('A fixed budget, divided more ways')

axes[1].loglog(n_frames_range, result['snr_per_frame'], 'o-', color='crimson')
axes[1].set_xlabel('number of time points in the movie'); axes[1].set_ylabel('SNR per frame (relative)')
axes[1].set_title('SNR falls as sqrt(photons per frame)')
plt.tight_layout(); plt.show()

for n, snr in zip(n_frames_range, result['snr_per_frame']):
    print(f'{n:>4d} frames  ->  relative SNR per frame: {snr:7.1f}')

🔍 **What just happened?** Quadrupling the number of time points (say, from 20 to 80) only
halves the per-frame SNR, not quarters it — the square-root relationship is forgiving compared to a
linear one, but the trend is still unavoidable and monotonic: there is no number of time points at
which more temporal resolution becomes free. A microscopist planning a time-lapse experiment is
always trading temporal resolution against per-frame image quality against total experiment
duration, against the same fixed photon budget, whether or not they ever write down the trade-off
this explicitly.

✏️ **Your turn.** A researcher needs at least a relative SNR of `300` per frame for a reliable
measurement. Using the printed table, what is the maximum number of time points this fixed
`total_budget` can support, and how does this number change if `total_budget` is doubled?

### 📘 Lecture notes — the photon budget connects every earlier module in this course's physics/instrumentation sequence

**Definition.** The photon budget is the total amount of excitation light a live specimen can
tolerate before unacceptable bleaching or phototoxicity; it must be divided among however many time
points, channels, and z-planes a time-lapse experiment collects, with per-measurement SNR falling as
the square root of the resulting photons-per-measurement.

**Worked example.** A multi-channel, multi-z-plane, long time-lapse experiment divides the same fixed
budget three ways at once — across channels, across z, and across time — which is exactly why real
live-cell imaging experiments are so often a three-way compromise between temporal resolution, axial
coverage, and per-frame image quality, not just a straightforward trade of "more time points for less
SNR."

**One memorable point.** *Every instrumentation choice in Modules 03-09 — NA, pinhole size,
contrast method, fluorophore brightness, sensor read noise — ultimately earns or spends against this
same, single fixed photon budget; there is no free parameter left over once the specimen's tolerance
for light is fixed.*

## 📌 Key takeaways

- Photon shot noise is a property of light itself, following Poisson statistics, with relative noise
  scaling as $1/\sqrt{N}$; it sets a noise floor no camera electronics can remove.
- A real camera's detection chain adds quantum efficiency, dark current, and read noise on top of
  shot noise, each independently simulable and visually identifiable.
- Read noise matters most at low photon counts; at high photon counts, shot noise dominates
  regardless of sensor quality.
- Bit-depth quantization becomes a visible problem (banding) when the quantization step is larger
  than the noise already present, and becomes pointless once it is much smaller.
- A live specimen's fixed photon budget links every imaging-parameter choice together: more time
  points, channels, or z-planes each mean fewer photons — and lower SNR — per individual
  measurement.

## ✅ Checkpoint

**1.** A microscopist doubles their camera's exposure time, expecting the image to look "twice as
good." Based on Section 1, what actually happens to the shot-noise-limited SNR, and why is the
microscopist's expectation off?

<details><summary>Answer</summary>
SNR improves by a factor of $\sqrt{2} \approx 1.41$, not 2, because shot-noise-limited SNR scales as
the square root of photon count (and exposure time), not linearly. Doubling exposure time doubles the
expected photon count, whose square root is $\sqrt{2}$, not 2.
</details>

**2.** A lab is deciding whether to invest in an expensive low-read-noise camera for a new bright,
high-signal bulk-fluorescence assay. Based on Section 3, is this upgrade likely to make a large
visible difference?

<details><summary>Answer</summary>
Probably not much of one. Section 3 showed that at high photon counts (a bright signal), shot noise
dominates over read noise for both sensor types, so the gap between a low- and high-read-noise sensor
becomes small. The upgrade matters far more for photon-starved applications (dim fluorescence,
localization microscopy, deep multiphoton imaging) than for a bright, high-signal assay.
</details>

**3.** A live-cell time-lapse is redesigned to capture twice as many time points within the same total
imaging session, using the same total photon budget. Using Section 5's framework, what happens to the
per-frame SNR, and is there any way to double the time points for free?

<details><summary>Answer</summary>
Per-frame SNR falls by a factor of $\sqrt{2} \approx 1.41$, since each frame now gets half the
photons. There is no way to double the number of time points "for free" within the same fixed photon
budget — more temporal resolution always costs per-frame image quality, unless the specimen's
tolerance for light (the budget itself) can somehow be increased, for instance by switching to a
less phototoxic imaging modality from Module 08.
</details>

## 🧠 Questions & discussion

- Pixel binning (combining several adjacent pixels into one larger effective pixel before or during
  readout) is a common strategy for photon-starved imaging. Using this module's framework, explain
  why binning improves SNR, and what it costs in exchange.
- This module treated quantum efficiency as a single fixed number. Real sensors often have
  wavelength-dependent quantum efficiency curves. How might this interact with the fluorophore and
  filter choices from Module 07?
- Given everything in this module, is it ever correct to say a microscope image is "too noisy" without
  specifying which stage of the detection chain — or which choice earlier in this sequence of
  modules — is actually responsible?

## 🚀 Bonus — going further

- Extend Section 2's stage-by-stage breakdown to compute each stage's measured standard deviation
  numerically (rather than only showing it visually), and compare it to the theoretical shot-noise,
  dark-current, and read-noise contributions added in quadrature.
- Simulate pixel binning explicitly (sum 2x2 blocks of a simulated noisy image) and verify the SNR
  improvement this produces, connecting it back to the photon-budget framework from Section 5.
- Combine this module's `sensor_noise_model` with Module 08's `tirf_evanescent_intensity` to simulate
  a complete TIRF acquisition — illumination, specimen, and camera — end to end.

## 📝 Lab note

```text
Module: 09 — Electronic Imaging: Detectors and the Digital Pipeline
Date:
The sensor read-noise spec and bit depth I would choose for MY own imaging question, and why:
One photon-budget trade-off I would now think through before designing a time-lapse experiment:
```

## 📚 References

1. Janesick, J.R. (2001). *Scientific Charge-Coupled Devices*. SPIE Press.
2. Pawley, J.B. (Ed.) (2006). *Handbook of Biological Confocal Microscopy* (3rd ed.). Springer.
3. Waters, J.C. (2009). Accuracy and precision in quantitative fluorescence microscopy. *Journal of
   Cell Biology*, 185(7), 1135-1148.
4. Murphy, D.B., Davidson, M.W. (2012). *Fundamentals of Light Microscopy and Electronic Imaging*
   (2nd ed.). Wiley-Blackwell. — a fuller treatment of camera technologies and the detection chain.

## ➡️ Next

Module 10 (a stretch module) puts this whole physics/instrumentation sequence into practice on a
single, realistic live-cell calibration problem: balancing photobleaching and phototoxicity against
the photon budget from this module, and correcting the uneven illumination every real microscope
field of view has to some degree.